
# OpportunityLens Bhutan

## Intelligent Opportunity Discovery and Assessment Agent

### Project Overview

OpportunityLens Bhutan is an intelligent AI agent designed to help students and young professionals discover, understand, and evaluate scholarships, internships, fellowships, conferences, and leadership programmes.

The system uses artificial intelligence, document processing, web research, and multi-step agent reasoning to assess opportunities based on an applicant's background, qualifications, interests, eligibility, and financial requirements.

### Main Objectives

1. Extract important information from opportunity announcements.
2. Discover opportunities through web research.
3. Evaluate applicant eligibility.
4. Analyse scholarship and programme funding coverage.
5. Identify missing requirements and application deadlines.
6. Maintain assessment history and user feedback.
7. Generate structured opportunity assessment reports.

### Technologies

- Python
- Ollama (Qwen 2.5 3B)
- LangGraph
- LangChain Core
- SQLite
- Pandas
- Streamlit
- Jupyter Notebook

### Target Users

Students and young professionals seeking educational, professional, and leadership opportunities, with particular attention to applicants from Bhutan.

### Important Limitation

OpportunityLens provides decision support, not guaranteed eligibility, verified authenticity, or selection outcomes. Applicants must independently confirm important information with official organisers.


In [1]:

# ============================================================
# CELL 1: IMPORT REQUIRED LIBRARIES
# Project: OpportunityLens Bhutan
# ============================================================

# Standard Python libraries
import os
import re
import json
import sqlite3
import uuid
from pathlib import Path
from datetime import datetime

# Data processing
import pandas as pd

# Data validation
from pydantic import BaseModel, Field

# PDF and Word document processing
from pypdf import PdfReader
from docx import Document

# Local AI model integration
from langchain_ollama import ChatOllama

# LangGraph agent framework
from langgraph.graph import StateGraph, START, END

# Display and visualization
import matplotlib.pyplot as plt

print("All project libraries imported successfully!")


All project libraries imported successfully!


In [2]:

# ============================================================
# CELL 2: PROJECT CONFIGURATION
# ============================================================

# Main project directory
PROJECT_DIR = Path.cwd().parent

# Project folders
DATA_DIR = PROJECT_DIR / "data"
UPLOAD_DIR = PROJECT_DIR / "uploads"
REPORT_DIR = PROJECT_DIR / "reports"
DATASET_DIR = PROJECT_DIR / "datasets"

# Database location
DATABASE_PATH = DATA_DIR / "opportunitylens.db"

# AI model configuration
MODEL_NAME = "qwen2.5:3b"
MODEL_TEMPERATURE = 0

# Create folders if they do not exist
for folder in [
    DATA_DIR,
    UPLOAD_DIR,
    REPORT_DIR,
    DATASET_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Display project configuration
print("PROJECT CONFIGURATION")
print("-" * 40)

print("Project directory:", PROJECT_DIR)
print("Database:", DATABASE_PATH)
print("Upload directory:", UPLOAD_DIR)
print("Reports directory:", REPORT_DIR)
print("Dataset directory:", DATASET_DIR)
print("AI model:", MODEL_NAME)


PROJECT CONFIGURATION
----------------------------------------
Project directory: d:\OpportunityLens_Bhutan
Database: d:\OpportunityLens_Bhutan\data\opportunitylens.db
Upload directory: d:\OpportunityLens_Bhutan\uploads
Reports directory: d:\OpportunityLens_Bhutan\reports
Dataset directory: d:\OpportunityLens_Bhutan\datasets
AI model: qwen2.5:3b



## 1. Local AI Model Connection

This section connects OpportunityLens Bhutan to the locally installed Qwen 2.5 3B model through Ollama.

The model will help extract information from opportunity announcements, interpret eligibility requirements, and generate assessment summaries.


In [3]:

# ============================================================
# CELL 3: CONNECT AND TEST OLLAMA
# ============================================================

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=MODEL_TEMPERATURE
)

response = llm.invoke(
    "Explain the purpose of an intelligent agent in three sentences."
)

print(response.content)


An intelligent agent is designed to understand and interpret human commands or queries, process the information, and provide appropriate responses or actions. Its primary purpose is to enhance user experience by automating tasks, providing personalized assistance, and improving efficiency in various applications such as customer service, healthcare, and home automation. Intelligent agents also play a crucial role in learning from interactions, improving their performance over time, and adapting to new situations, thereby becoming more effective and user-friendly.



## 2. Opportunity Document Processing

This component accepts opportunity announcements in PDF, Word, and text formats.

It extracts the document text and uses the local AI model to identify important information, including deadlines, eligibility, funding, application requirements, and programme details.


In [4]:

# ============================================================
# CELL 4: DOCUMENT TEXT EXTRACTION
# ============================================================

def extract_document_text(file_path):
    """Extract text from PDF, DOCX, or TXT documents."""

    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    extension = path.suffix.lower()

    if extension == ".pdf":
        reader = PdfReader(str(path))
        text = "\n".join(
            page.extract_text() or "" for page in reader.pages
        )

    elif extension == ".docx":
        document = Document(str(path))
        text = "\n".join(
            paragraph.text for paragraph in document.paragraphs
        )

    elif extension in [".txt", ".md"]:
        text = path.read_text(encoding="utf-8")

    else:
        raise ValueError(
            "Supported formats: PDF, DOCX, TXT and MD."
        )

    if not text.strip():
        raise ValueError(
            "No readable text was extracted. "
            "Scanned PDFs may require OCR."
        )

    return text


print("Document extraction function is ready.")


Document extraction function is ready.


In [5]:

# ============================================================
# CELL 5: JSON RESPONSE PROCESSING
# ============================================================

def parse_json_response(response_text):
    """Extract a JSON object from an AI response."""

    cleaned = response_text.strip()

    # Remove Markdown code fences if present
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    # Try parsing the entire response
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Find the first complete JSON object
    start = cleaned.find("{")

    if start == -1:
        raise ValueError("No JSON object found in AI response.")

    decoder = json.JSONDecoder()
    result, _ = decoder.raw_decode(cleaned[start:])

    return result


def ask_ai_for_json(prompt):
    """Ask the local AI model to return structured JSON."""

    response = llm.invoke(
        prompt + "\n\nReturn valid JSON only. Do not use Markdown."
    )

    return parse_json_response(response.content)


print("JSON processing functions are ready.")


JSON processing functions are ready.


In [6]:

# ============================================================
# CELL 6: OPPORTUNITY INFORMATION EXTRACTION
# ============================================================

def extract_opportunity_information(document_text):

    prompt = f"""
You are an opportunity information extraction assistant.

Extract information from the announcement below.

Return a JSON object with these fields:

{{
    "title": "",
    "organizer": "",
    "opportunity_type": "",
    "location": "",
    "country": "",
    "description": "",
    "eligibility": [],
    "academic_requirements": [],
    "age_requirements": [],
    "nationality_requirements": [],
    "funding_type": "",
    "funding_coverage": [],
    "application_fee": "",
    "deadline": "",
    "start_date": "",
    "duration": "",
    "required_documents": [],
    "application_process": [],
    "official_website": "",
    "missing_information": []
}}

Rules:
1. Only extract information supported by the announcement.
2. Do not invent deadlines, fees, funding or eligibility rules.
3. Use "Not specified" when information is unavailable.
4. Use missing_information to identify important missing details.
5. Preserve the meaning of the original announcement.

ANNOUNCEMENT:
{document_text[:18000]}
"""

    data = ask_ai_for_json(prompt)

    required_fields = [
        "title", "organizer", "opportunity_type",
        "location", "description", "eligibility",
        "funding_type", "funding_coverage",
        "deadline", "required_documents"
    ]

    for field in required_fields:
        data.setdefault(field, "Not specified")

    return data


print("Opportunity extraction function is ready.")


Opportunity extraction function is ready.



## 3. Applicant Profiling and Web Research

The applicant profile stores educational background, nationality, qualifications, skills, interests, and financial preferences.

The web research component searches for additional information about an opportunity. Search results are treated as supporting leads, not proof of authenticity.


In [7]:

# ============================================================
# CELL 7: APPLICANT PROFILE
# ============================================================

applicant_profile = {
    "name": "Sample Applicant",
    "nationality": "Bhutanese",
    "country": "Bhutan",
    "education_level": "Undergraduate",
    "field_of_study": "Data Science and Data Analytics",
    "academic_year": "Final Year",
    "skills": [
        "Python",
        "SQL",
        "Data Analysis",
        "Machine Learning",
        "Excel"
    ],
    "interests": [
        "Scholarships",
        "Internships",
        "Artificial Intelligence",
        "Leadership",
        "Sustainability"
    ],
    "experience": [
        "Academic projects",
        "Volunteer activities",
        "Leadership experience"
    ],
    "financial_preference": "Fully funded or partially funded",
    "preferred_locations": [
        "Bhutan",
        "Asia",
        "Worldwide"
    ]
}

print(json.dumps(applicant_profile, indent=4))


{
    "name": "Sample Applicant",
    "nationality": "Bhutanese",
    "country": "Bhutan",
    "education_level": "Undergraduate",
    "field_of_study": "Data Science and Data Analytics",
    "academic_year": "Final Year",
    "skills": [
        "Python",
        "SQL",
        "Data Analysis",
        "Machine Learning",
        "Excel"
    ],
    "interests": [
        "Scholarships",
        "Internships",
        "Artificial Intelligence",
        "Leadership",
        "Sustainability"
    ],
    "experience": [
        "Academic projects",
        "Volunteer activities",
        "Leadership experience"
    ],
    "financial_preference": "Fully funded or partially funded",
    "preferred_locations": [
        "Bhutan",
        "Asia",
        "Worldwide"
    ]
}


In [8]:

# ============================================================
# CELL 8: WEB RESEARCH TOOL
# ============================================================

from ddgs import DDGS


def search_web(query, max_results=5):
    """
    Search the web for relevant opportunity information.
    Returns titles, URLs and snippets where available.
    """

    results = []

    try:
        with DDGS() as search_engine:
            search_results = search_engine.text(
                query,
                max_results=max_results
            )

            for item in search_results:
                results.append({
                    "title": item.get("title", ""),
                    "url": item.get("href", ""),
                    "snippet": item.get("body", "")
                })

    except Exception as error:
        print("Web search issue:", error)

    return results


def research_opportunity(opportunity):
    """Search for information related to an opportunity."""

    title = opportunity.get("title", "")
    organizer = opportunity.get("organizer", "")

    queries = [
        f'"{title}" "{organizer}" official',
        f'"{title}" eligibility deadline funding',
        f'"{organizer}" official website opportunity'
    ]

    all_results = []
    seen_urls = set()

    for query in queries:

        results = search_web(query, max_results=5)

        for result in results:
            url = result.get("url", "")

            if url and url not in seen_urls:
                seen_urls.add(url)
                result["search_query"] = query
                all_results.append(result)

    return all_results


print("Web research functions are ready.")


Web research functions are ready.



## 4. Eligibility and Funding Assessment

This component compares the applicant's profile with the opportunity requirements.

It distinguishes between eligible, potentially eligible, not eligible, and unknown cases.

Funding is assessed separately so that the system does not incorrectly describe a partially funded opportunity as fully funded.


In [9]:

# ============================================================
# CELL 9: ELIGIBILITY ASSESSMENT
# ============================================================

def assess_eligibility(applicant, opportunity):

    prompt = f"""
You are an eligibility assessment assistant.

Compare the applicant profile with the opportunity requirements.

APPLICANT:
{json.dumps(applicant, indent=2)}

OPPORTUNITY:
{json.dumps(opportunity, indent=2)}

Return JSON using this structure:

{{
    "eligibility_status": "Eligible, Possibly Eligible, Not Eligible, or Unknown",
    "eligibility_score": 0,
    "matching_requirements": [],
    "unmet_requirements": [],
    "uncertain_requirements": [],
    "missing_applicant_information": [],
    "explanation": ""
}}

Rules:
- Never assume an unstated requirement.
- Do not treat missing information as proof of eligibility.
- Do not mark someone ineligible without a clear unmet mandatory condition.
- Score from 0 to 100 as an indication of profile matching,
  not a probability of selection.
- Explain uncertainty honestly.
"""

    result = ask_ai_for_json(prompt)

    result.setdefault("eligibility_status", "Unknown")
    result.setdefault("eligibility_score", 0)
    result.setdefault("matching_requirements", [])
    result.setdefault("unmet_requirements", [])
    result.setdefault("uncertain_requirements", [])
    result.setdefault("missing_applicant_information", [])
    result.setdefault("explanation", "")

    return result


print("Eligibility assessment function is ready.")


Eligibility assessment function is ready.


In [10]:

# ============================================================
# CELL 10: FUNDING ASSESSMENT
# ============================================================

def assess_funding(opportunity):

    prompt = f"""
Analyse the funding information in this opportunity.

OPPORTUNITY:
{json.dumps(opportunity, indent=2)}

Return JSON:

{{
    "funding_category": "Fully Funded, Partially Funded, Self Funded, or Unknown",
    "covered_expenses": [],
    "uncovered_expenses": [],
    "financial_risks": [],
    "application_fee": "",
    "funding_explanation": ""
}}

Rules:
- Only classify funding based on available evidence.
- Do not assume that accommodation, flights, meals or visas are covered.
- Distinguish between confirmed and unspecified expenses.
- If information is insufficient, use Unknown.
"""

    result = ask_ai_for_json(prompt)

    result.setdefault("funding_category", "Unknown")
    result.setdefault("covered_expenses", [])
    result.setdefault("uncovered_expenses", [])
    result.setdefault("financial_risks", [])
    result.setdefault("application_fee", "Not specified")
    result.setdefault("funding_explanation", "")

    return result


print("Funding assessment function is ready.")


Funding assessment function is ready.



## 5. Persistent Memory and Feedback

OpportunityLens uses SQLite to maintain assessment history and user feedback.

This allows the system to retrieve previous assessments and understand which opportunities the applicant found useful.

Feedback is stored as user feedback and is not automatically treated as verified eligibility information.


In [11]:

# ============================================================
# CELL 11: DATABASE INITIALIZATION
# ============================================================

def initialize_database():

    connection = sqlite3.connect(DATABASE_PATH)
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS assessments (
            id TEXT PRIMARY KEY,
            opportunity_title TEXT,
            applicant_name TEXT,
            eligibility_status TEXT,
            eligibility_score REAL,
            funding_category TEXT,
            assessment_json TEXT,
            report_path TEXT,
            created_at TEXT
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            assessment_id TEXT,
            rating INTEGER,
            comment TEXT,
            created_at TEXT
        )
    """)

    connection.commit()
    connection.close()

    print("OpportunityLens database initialized.")


initialize_database()


OpportunityLens database initialized.


In [12]:

# ============================================================
# CELL 12: SAVE AND RETRIEVE ASSESSMENTS
# ============================================================

def save_assessment(result):

    assessment_id = result["assessment_id"]

    connection = sqlite3.connect(DATABASE_PATH)

    connection.execute("""
        INSERT OR REPLACE INTO assessments
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        assessment_id,
        result["opportunity"].get("title", "Unknown"),
        result["applicant"].get("name", "Unknown"),
        result["eligibility"].get("eligibility_status", "Unknown"),
        result["eligibility"].get("eligibility_score", 0),
        result["funding"].get("funding_category", "Unknown"),
        json.dumps(result, default=str),
        result.get("report_path", ""),
        datetime.now().isoformat()
    ))

    connection.commit()
    connection.close()

    print("Assessment saved successfully.")


def get_assessment_history():

    connection = sqlite3.connect(DATABASE_PATH)

    history = pd.read_sql_query(
        "SELECT * FROM assessments ORDER BY created_at DESC",
        connection
    )

    connection.close()

    return history


def save_feedback(assessment_id, rating, comment):

    if not 1 <= rating <= 5:
        raise ValueError("Rating must be between 1 and 5.")

    connection = sqlite3.connect(DATABASE_PATH)

    connection.execute("""
        INSERT INTO feedback
        (assessment_id, rating, comment, created_at)
        VALUES (?, ?, ?, ?)
    """, (
        assessment_id,
        rating,
        comment,
        datetime.now().isoformat()
    ))

    connection.commit()
    connection.close()

    print("Feedback saved.")


print("Memory and feedback functions are ready.")


Memory and feedback functions are ready.



## 6. LangGraph Agent Architecture

The agent follows a multi-step workflow:

1. Planning: determine the information needed.
2. Extraction: understand the opportunity announcement.
3. Research: gather additional supporting evidence.
4. Assessment: evaluate eligibility and funding.
5. Reflection: check whether important information is missing.
6. Reporting: prepare the final assessment.

The reflection stage can send the agent back to research, subject to a maximum number of attempts.


In [13]:

# ============================================================
# CELL 13: AGENT STATE
# ============================================================

from typing import TypedDict, Any


class OpportunityAgentState(TypedDict, total=False):

    applicant: dict
    document_text: str
    file_path: str

    opportunity: dict
    research_results: list

    eligibility: dict
    funding: dict

    plan: list
    reflection: str

    research_attempt: int
    max_research_attempts: int

    assessment_id: str
    report: str
    report_path: str

    workflow_log: list


print("Agent state structure created.")


Agent state structure created.


In [14]:

# ============================================================
# CELL 14: AGENT PLANNING NODE
# ============================================================

def planning_node(state):

    plan = [
        "Extract opportunity information",
        "Research the opportunity",
        "Assess applicant eligibility",
        "Analyse funding coverage",
        "Reflect on evidence quality",
        "Generate assessment report"
    ]

    log = state.get("workflow_log", [])

    log.append({
        "step": "Planning",
        "time": datetime.now().isoformat(),
        "message": "Agent created its execution plan."
    })

    return {
        "plan": plan,
        "workflow_log": log,
        "research_attempt": 0
    }


# ============================================================
# CELL 15: EXTRACTION NODE
# ============================================================

def extraction_node(state):

    document_text = state.get("document_text", "")

    if state.get("file_path"):
        document_text = extract_document_text(
            state["file_path"]
        )

    if not document_text.strip():
        raise ValueError("No opportunity announcement provided.")

    opportunity = extract_opportunity_information(
        document_text
    )

    log = state.get("workflow_log", [])

    log.append({
        "step": "Extraction",
        "time": datetime.now().isoformat(),
        "message": "Opportunity information extracted."
    })

    return {
        "document_text": document_text,
        "opportunity": opportunity,
        "workflow_log": log
    }


# ============================================================
# CELL 16: RESEARCH NODE
# ============================================================

def research_node(state):

    opportunity = state["opportunity"]

    results = research_opportunity(opportunity)

    log = state.get("workflow_log", [])

    log.append({
        "step": "Research",
        "time": datetime.now().isoformat(),
        "message": f"Research attempt completed with {len(results)} results."
    })

    return {
        "research_results": results,
        "research_attempt": state.get("research_attempt", 0) + 1,
        "workflow_log": log
    }


# ============================================================
# CELL 17: ASSESSMENT NODE
# ============================================================

def assessment_node(state):

    applicant = state["applicant"]
    opportunity = state["opportunity"]

    eligibility = assess_eligibility(
        applicant,
        opportunity
    )

    funding = assess_funding(opportunity)

    log = state.get("workflow_log", [])

    log.append({
        "step": "Assessment",
        "time": datetime.now().isoformat(),
        "message": "Eligibility and funding assessments completed."
    })

    return {
        "eligibility": eligibility,
        "funding": funding,
        "workflow_log": log
    }


# ============================================================
# CELL 18: REFLECTION NODE
# ============================================================

def reflection_node(state):

    opportunity = state.get("opportunity", {})
    research = state.get("research_results", [])

    missing_fields = []

    for field in ["title", "organizer", "deadline"]:
        value = opportunity.get(field, "Not specified")

        if not value or value == "Not specified":
            missing_fields.append(field)

    if not research:
        reflection = (
            "No external search results were found. "
            "The assessment requires independent verification."
        )
    elif missing_fields:
        reflection = (
            "Some important opportunity details are missing: "
            + ", ".join(missing_fields)
        )
    else:
        reflection = (
            "The agent completed its assessment. "
            "Applicants should independently verify the evidence."
        )

    log = state.get("workflow_log", [])

    log.append({
        "step": "Reflection",
        "time": datetime.now().isoformat(),
        "message": reflection
    })

    return {
        "reflection": reflection,
        "workflow_log": log
    }


# ============================================================
# CELL 19: CONDITIONAL ROUTING
# ============================================================

def route_after_reflection(state):

    attempts = state.get("research_attempt", 0)
    maximum = state.get("max_research_attempts", 2)

    results = state.get("research_results", [])

    # Retry if no research results were obtained.
    if not results and attempts < maximum:
        return "research"

    return "report"


print("Agent nodes and routing functions are ready.")


Agent nodes and routing functions are ready.



## 7. Assessment Report Generation

The final report combines extracted opportunity details, research evidence, eligibility results, funding information, and the agent's reflection.

It is saved as a Markdown report in the reports folder.


In [15]:

# ============================================================
# CELL 20: REPORT GENERATION NODE
# ============================================================

def report_node(state):

    opportunity = state["opportunity"]
    eligibility = state["eligibility"]
    funding = state["funding"]

    assessment_id = state.get(
        "assessment_id",
        str(uuid.uuid4())
    )

    evidence = state.get("research_results", [])

    report = f"""
# OpportunityLens Bhutan
## Opportunity Assessment Report

Assessment ID: {assessment_id}

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}

---

## 1. Opportunity Information

**Title:** {opportunity.get("title")}

**Organizer:** {opportunity.get("organizer")}

**Type:** {opportunity.get("opportunity_type")}

**Location:** {opportunity.get("location")}

**Deadline:** {opportunity.get("deadline")}

**Description:** {opportunity.get("description")}

## 2. Eligibility Assessment

**Status:** {eligibility.get("eligibility_status")}

**Matching Score:** {eligibility.get("eligibility_score")}/100

**Matching Requirements:**
{json.dumps(eligibility.get("matching_requirements", []), indent=2)}

**Unmet Requirements:**
{json.dumps(eligibility.get("unmet_requirements", []), indent=2)}

**Uncertain Requirements:**
{json.dumps(eligibility.get("uncertain_requirements", []), indent=2)}

**Explanation:**
{eligibility.get("explanation")}

## 3. Funding Assessment

**Funding Category:** {funding.get("funding_category")}

**Covered Expenses:**
{json.dumps(funding.get("covered_expenses", []), indent=2)}

**Uncovered Expenses:**
{json.dumps(funding.get("uncovered_expenses", []), indent=2)}

**Financial Risks:**
{json.dumps(funding.get("financial_risks", []), indent=2)}

**Explanation:**
{funding.get("funding_explanation")}

## 4. Research Evidence

"""

    if evidence:

        for index, item in enumerate(evidence, start=1):

            report += f"""
### Source {index}

Title: {item.get("title")}

URL: {item.get("url")}

Information: {item.get("snippet")}

"""

    else:

        report += "No external research results were obtained.\n"

    report += f"""
## 5. Agent Reflection

{state.get("reflection", "")}

## 6. Important Disclaimer

This report provides AI-generated decision support.
It does not guarantee eligibility, authenticity, funding,
or selection. Verify all important information directly
with the official organiser before applying or paying fees.
"""

    report_path = REPORT_DIR / f"{assessment_id}.md"

    report_path.write_text(
        report,
        encoding="utf-8"
    )

    log = state.get("workflow_log", [])

    log.append({
        "step": "Reporting",
        "time": datetime.now().isoformat(),
        "message": "Assessment report generated."
    })

    return {
        "assessment_id": assessment_id,
        "report": report,
        "report_path": str(report_path),
        "workflow_log": log
    }


print("Report generation function is ready.")


Report generation function is ready.


In [16]:

# ============================================================
# CELL 21: COMPILE LANGGRAPH WORKFLOW
# ============================================================

workflow = StateGraph(OpportunityAgentState)

# Register agent nodes
workflow.add_node("planning", planning_node)
workflow.add_node("extraction", extraction_node)
workflow.add_node("research", research_node)
workflow.add_node("assessment", assessment_node)
workflow.add_node("reflection", reflection_node)
workflow.add_node("report", report_node)

# Define workflow connections
workflow.add_edge(START, "planning")
workflow.add_edge("planning", "extraction")
workflow.add_edge("extraction", "research")
workflow.add_edge("research", "assessment")
workflow.add_edge("assessment", "reflection")

# Conditional reflection and research loop
workflow.add_conditional_edges(
    "reflection",
    route_after_reflection,
    {
        "research": "research",
        "report": "report"
    }
)

workflow.add_edge("report", END)

# Compile agent
opportunity_agent = workflow.compile()

print("OpportunityLens LangGraph agent compiled successfully!")


OpportunityLens LangGraph agent compiled successfully!



## 8. Execute OpportunityLens Bhutan

The following function accepts an opportunity announcement and applicant profile, executes the LangGraph workflow, stores the assessment, and returns a complete report.


In [17]:

# ============================================================
# CELL 22: MAIN AGENT EXECUTION FUNCTION
# ============================================================

def run_opportunity_agent(
    applicant,
    document_text=None,
    file_path=None
):

    if not document_text and not file_path:
        raise ValueError(
            "Provide announcement text or a document path."
        )

    initial_state = {
        "applicant": applicant,
        "document_text": document_text or "",
        "file_path": file_path or "",
        "research_results": [],
        "research_attempt": 0,
        "max_research_attempts": 2,
        "workflow_log": [],
        "assessment_id": str(uuid.uuid4())
    }

    print("Starting OpportunityLens Bhutan...")
    print("-" * 50)

    result = opportunity_agent.invoke(
        initial_state
    )

    save_assessment(result)

    print("\nAgent workflow completed.")

    return result


print("Main execution function is ready.")


Main execution function is ready.



## 9. Prototype Testing

A fictional opportunity announcement is used to test document extraction, research, eligibility assessment, funding analysis, and report generation.

This sample is for software testing only and is not a real scholarship or programme.


In [18]:

# ============================================================
# CELL 23: SAMPLE OPPORTUNITY ANNOUNCEMENT
# ============================================================

sample_announcement = """
GLOBAL DATA SCIENCE FELLOWSHIP 2027

Organizer: Example International Education Foundation

Opportunity Type: Fellowship

Location: Singapore

Description:
A fictional international fellowship designed for
undergraduate students interested in data science,
artificial intelligence, and sustainable development.

Eligibility:
- Applicants must be currently enrolled undergraduate students.
- Applicants from all nationalities may apply.
- Applicants must demonstrate an interest in data science.
- Applicants must have basic programming knowledge.

Funding:
The fictional programme provides accommodation and
training costs. International travel and visa expenses
are not specified.

Application Fee:
No application fee is stated.

Deadline:
30 January 2027.

Required Documents:
- Curriculum Vitae
- Academic transcript
- Personal statement

Application Process:
Submit the required documents through the organiser's
application portal.

Official Website:
Not specified.
"""

print("Sample announcement prepared.")


Sample announcement prepared.


In [19]:

# ============================================================
# CELL 24: RUN THE AGENT
# ============================================================

test_result = run_opportunity_agent(
    applicant=applicant_profile,
    document_text=sample_announcement
)

print("\nOPPORTUNITY:")
print(json.dumps(test_result["opportunity"], indent=2))

print("\nELIGIBILITY:")
print(json.dumps(test_result["eligibility"], indent=2))

print("\nFUNDING:")
print(json.dumps(test_result["funding"], indent=2))


Starting OpportunityLens Bhutan...
--------------------------------------------------
Web search issue: No results found.
Web search issue: No results found.
Web search issue: error sending request for url (https://www.startpage.com/) > operation timed out
Web search issue: No results found.
Web search issue: No results found.
Assessment saved successfully.

Agent workflow completed.

OPPORTUNITY:
{
  "title": "GLOBAL DATA SCIENCE FELLOWSHIP 2027",
  "organizer": "Example International Education Foundation",
  "opportunity_type": "Fellowship",
  "location": "Singapore",
  "country": "Singapore",
  "description": "A fictional international fellowship designed for undergraduate students interested in data science, artificial intelligence, and sustainable development.",
  "eligibility": [
    "Currently enrolled undergraduate students",
    "Applicants from all nationalities may apply",
    "Applicants must demonstrate an interest in data science",
    "Applicants must have basic programm

In [20]:

# ============================================================
# CELL 25: DISPLAY FINAL REPORT
# ============================================================

from IPython.display import display, Markdown

display(
    Markdown(test_result["report"])
)

print("\nReport saved at:")
print(test_result["report_path"])



# OpportunityLens Bhutan
## Opportunity Assessment Report

Assessment ID: 57e82e2d-8828-46a3-a7c2-c604aa9bca69

Generated: 2026-11-14 11:36

---

## 1. Opportunity Information

**Title:** GLOBAL DATA SCIENCE FELLOWSHIP 2027

**Organizer:** Example International Education Foundation

**Type:** Fellowship

**Location:** Singapore

**Deadline:** 30 January 2027

**Description:** A fictional international fellowship designed for undergraduate students interested in data science, artificial intelligence, and sustainable development.

## 2. Eligibility Assessment

**Status:** Eligible, Possibly Eligible, Not Eligible, or Unknown

**Matching Score:** 50/100

**Matching Requirements:**
[
  "Currently enrolled undergraduate students",
  "Applicants from all nationalities may apply",
  "Applicants must demonstrate an interest in data science",
  "Applicants must have basic programming knowledge"
]

**Unmet Requirements:**
[
  "Applicants must have basic programming knowledge"
]

**Uncertain Requirements:**
[]

**Explanation:**
The applicant is eligible based on their education level and field of study. However, they have not provided any specific programming knowledge or experience, which is a requirement for the opportunity. The applicant's profile matches the opportunity in terms of nationality, field of study, and interests, but the lack of specific programming knowledge prevents a higher score.

## 3. Funding Assessment

**Funding Category:** Unknown

**Covered Expenses:**
[]

**Uncovered Expenses:**
[]

**Financial Risks:**
[]

**Explanation:**


## 4. Research Evidence


### Source 1

Title: Supporting non-state providers in basic education service ...

URL: https://archive.org/download/ERIC_ED508616/ERIC_ED508616.pdf

Information: The Consortium for Educational Access, Transitions and Equity (CREATE) is a Research Programme Consortium supported by the UK Department for International Development (DFID). Its purpose is to undertake research designed to improve access to basic education in developing countries. It seeks to achieve this through generating new knowledge and encouraging its application through effective ...


### Source 2

Title: Guidebook for planning education in emergencies and ... - UNESCO

URL: https://unesdoc.unesco.org/ark:/48223/pf0000190223

Information: Explore UNESCO's initiatives, publications, and resources to promote education, science, culture, and communication worldwide.


### Source 3

Title: Development and psychometric testing of the perception of ...

URL: https://onlinelibrary.wiley.com/doi/10.1111/jan.15726

Information: May 30, 2023 · Moreover, nursing is defined in national descriptions of the nursing profession, such as official governmental guidelines or laws that set national standards for the profession.


### Source 4

Title: The Ministry in action - France Diplomatie

URL: https://www.diplomatie.gouv.fr/en/the-ministry-in-action

Information: The Ministry in action The Ministry for Europe and Foreign Affairs represents, defends and promotes the interests of France and French nationals in all areas in foreign countries and international organizations.


### Source 5

Title: Policy for Research Promotion - msruas.ac.in

URL: https://www.msruas.ac.in/uploads/pdf/Research-Promotion-Policy.pdf

Information: Table of Content Content Page No. Research @ RUAS Research Promotion and Management - Research Innovation and Council Aim of the Research Promotion Policy 04 1. MoUs and Non-Disclosure Agreement 05 2. Incentives 06 3. Fee Concession to Employees 07 4. Seed Money to Initiate Research 07 5. Research Centers and Center of Excellence 08 6. Support for Organizing Events (Conference, Seminars ...


## 5. Agent Reflection

The agent completed its assessment. Applicants should independently verify the evidence.

## 6. Important Disclaimer

This report provides AI-generated decision support.
It does not guarantee eligibility, authenticity, funding,
or selection. Verify all important information directly
with the official organiser before applying or paying fees.



Report saved at:
d:\OpportunityLens_Bhutan\reports\57e82e2d-8828-46a3-a7c2-c604aa9bca69.md



## 10. User Feedback and Assessment History

Users can rate an assessment and provide comments. OpportunityLens stores feedback and assessment history for future analysis.


In [21]:

# ============================================================
# CELL 26: SAVE USER FEEDBACK
# ============================================================

save_feedback(
    assessment_id=test_result["assessment_id"],
    rating=4,
    comment="The assessment was useful, but more official sources would improve confidence."
)

print("Feedback demonstration completed.")


Feedback saved.
Feedback demonstration completed.


In [22]:

# ============================================================
# CELL 27: VIEW ASSESSMENT HISTORY
# ============================================================

history = get_assessment_history()

display(history)


,id,opportunity_title,applicant_name,eligibility_status,eligibility_score,funding_category,assessment_json,report_path,created_at
0,57e82e2d-8828-46a3-a7c2-c604aa9bca69,GLOBAL DATA SCIENCE FELLOWSHIP 2027,Sample Applicant,"Eligible, Possibly Eligible, Not Eligible, or ...",50.0,Unknown,"{""applicant"": {""name"": ""Sample Applicant"", ""na...",d:\OpportunityLens_Bhutan\reports\57e82e2d-882...,2026-11-14T11:36:50.139636



## 11. Agent Evaluation

The system will be evaluated using labelled test cases.

Evaluation will consider:
- Eligibility classification performance.
- Accuracy of extracted opportunity details.
- Funding classification.
- Research availability.
- Workflow completion.

Actual performance must be measured using manually verified ground-truth labels rather than assumed.


In [23]:

# ============================================================
# CELL 28: CREATE EVALUATION DATASET TEMPLATE
# ============================================================

evaluation_template = pd.DataFrame([
    {
        "case_id": "TEST001",
        "expected_eligibility": "Eligible",
        "predicted_eligibility": "",
        "expected_funding": "Fully Funded",
        "predicted_funding": "",
        "expected_title": "",
        "predicted_title": ""
    },
    {
        "case_id": "TEST002",
        "expected_eligibility": "Not Eligible",
        "predicted_eligibility": "",
        "expected_funding": "Partially Funded",
        "predicted_funding": "",
        "expected_title": "",
        "predicted_title": ""
    },
    {
        "case_id": "TEST003",
        "expected_eligibility": "Unknown",
        "predicted_eligibility": "",
        "expected_funding": "Unknown",
        "predicted_funding": "",
        "expected_title": "",
        "predicted_title": ""
    }
])

evaluation_path = DATASET_DIR / "evaluation_template.csv"

evaluation_template.to_csv(
    evaluation_path,
    index=False
)

print("Evaluation template created:")
print(evaluation_path)

display(evaluation_template)


Evaluation template created:
d:\OpportunityLens_Bhutan\datasets\evaluation_template.csv


,case_id,expected_eligibility,predicted_eligibility,expected_funding,predicted_funding,expected_title,predicted_title
0,TEST001,Eligible,,Fully Funded,,,
1,TEST002,Not Eligible,,Partially Funded,,,
2,TEST003,Unknown,,Unknown,,,


In [24]:

# ============================================================
# CELL 29: ELIGIBILITY CLASSIFICATION METRICS
# ============================================================

def calculate_classification_metrics(
    actual,
    predicted
):

    labels = sorted(set(actual) | set(predicted))

    results = []

    for label in labels:

        tp = sum(
            a == label and p == label
            for a, p in zip(actual, predicted)
        )

        fp = sum(
            a != label and p == label
            for a, p in zip(actual, predicted)
        )

        fn = sum(
            a == label and p != label
            for a, p in zip(actual, predicted)
        )

        precision = (
            tp / (tp + fp)
            if tp + fp else 0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn else 0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        results.append({
            "class": label,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        })

    return pd.DataFrame(results)


def evaluate_agent_predictions(csv_path):

    dataset = pd.read_csv(csv_path)

    required = [
        "expected_eligibility",
        "predicted_eligibility"
    ]

    if not all(column in dataset.columns for column in required):
        raise ValueError("Evaluation CSV is missing required columns.")

    dataset = dataset.dropna(subset=required)

    actual = dataset["expected_eligibility"].astype(str).tolist()
    predicted = dataset["predicted_eligibility"].astype(str).tolist()

    if not actual:
        raise ValueError("No labelled predictions are available.")

    accuracy = sum(
        a == p for a, p in zip(actual, predicted)
    ) / len(actual)

    metrics = calculate_classification_metrics(
        actual,
        predicted
    )

    print(f"Overall accuracy: {accuracy:.2%}")

    return metrics


print("Evaluation functions are ready.")


Evaluation functions are ready.


In [25]:

# ============================================================
# CELL 30: VISUALIZE LANGGRAPH WORKFLOW
# ============================================================

try:

    display(
        opportunity_agent.get_graph().draw_mermaid_png()
    )

except Exception as error:

    print("Graph image rendering unavailable:", error)

    print(
        opportunity_agent.get_graph().draw_mermaid()
    )


b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\xe0\x00\x00\x02\xa5\x08\x02\x00\x00\x00]Ew@\x00\x00\x10\x00IDATx\x9c\xec\x9d\x07`\x14E\x17\xc7\xe7z\xc9\x91\x0e)\x04H\xe8\xbd\x19\x8a\x88\x80t\x15\x94& \xa2\x02*Mi\x02\n*\x8aTE\x9aR>EP\x04\xa4\xf7\xae\x08"\x02\xa2\xa0\xf4\x8e\x10\x08\x81\x04\xd2\x93\xeb\xed{\xb9\x81es\xb9\xc4\\\x02\xdc\xec\xe6\xfd\xe4\xbboo\xb6\xdcN\xf6\xbfo\xde\xcc\xce\xbe\'w:\x9d\x04AXEJ\x10\x84aP\xa0\x08\xd3\xa0@\x11\xa6\x91\x93\x87\r:\xb5\xac!\x91H\x88`y8\x02\xe5D\x99w\x01\xf1!T\x9a\xf4Zp2\x15\x96^\x8b%P\xbe\x1c)\xdc\xb2\xdb\x06\xc8\xe3\x87\n\x91\xfb\xe4\x7f\xe5o\xc08E\x14(_\x8b\x1cqg\xf5\xd7\xcf\x9b2\xee\xda\xccF\x07`\xb3\xe4\xf9\x8bH\x1dN\x87\xbb\xd7+\x91\xc0\xb1\xe0?\xb7r\x87\x07\xff\xd8\xe9\xc8\xd9ZBr\xfe\xc7?\x82\xa7\xc3\x12b\'D\xe6p\xda\xa5\x12Y\xee\xdfs\xe4\xf9-\xfaspl\xa8Q!N\xe3^\xb9\xc4\xf5)s\xaf\x8b\xcbZ\xe5\xdd\xe1\xdeIz\xfeu\xbb\xdbqr\x9d-\xe0a\x17"\x959\x1d\xf6|\x15&U8\x95J\xa9B#\r\x0e\x93Um\xa0\r)\xab\x92\xe4\xe1\xfe93-S\x89\xb7F\xceM\x9a \xc4_V&\'\\6\

In [26]:

# ============================================================
# CELL 31: PROJECT SUMMARY
# ============================================================

print("=" * 55)
print("OPPORTUNITYLENS BHUTAN - PROJECT SUMMARY")
print("=" * 55)

print("AI Model:", MODEL_NAME)
print("Database:", DATABASE_PATH)
print("Reports Folder:", REPORT_DIR)

print("\nAssessment History:")

history = get_assessment_history()

print("Total assessments:", len(history))

print("\nLatest assessment:")

if not history.empty:
    display(history.head(5))
else:
    print("No assessments recorded yet.")

print("\nGenerated Reports:")

for report_file in REPORT_DIR.glob("*.md"):
    print(report_file.name)

print("\nProject execution summary completed.")


OPPORTUNITYLENS BHUTAN - PROJECT SUMMARY
AI Model: qwen2.5:3b
Database: d:\OpportunityLens_Bhutan\data\opportunitylens.db
Reports Folder: d:\OpportunityLens_Bhutan\reports

Assessment History:
Total assessments: 1

Latest assessment:


,id,opportunity_title,applicant_name,eligibility_status,eligibility_score,funding_category,assessment_json,report_path,created_at
0,57e82e2d-8828-46a3-a7c2-c604aa9bca69,GLOBAL DATA SCIENCE FELLOWSHIP 2027,Sample Applicant,"Eligible, Possibly Eligible, Not Eligible, or ...",50.0,Unknown,"{""applicant"": {""name"": ""Sample Applicant"", ""na...",d:\OpportunityLens_Bhutan\reports\57e82e2d-882...,2026-11-14T11:36:50.139636



Generated Reports:
57e82e2d-8828-46a3-a7c2-c604aa9bca69.md

Project execution summary completed.
